In [2]:
# Import packages
import requests
import io
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
from datetime import date, timedelta
from IPython.display import display, Markdown

In [5]:
###################
#SET E-MAIL HEADER#
###################

# This cell will serve as a header in your email. You can add some information here that may be useful for providing context. 
# If you want to change the actual text that appears, feel free to edit the "md_text" variable directly.

# Input a title of your choosing here.
title = "311 Analysis"

# Write a brief description about the analysis.
description = """
This script will take 311 service requests and tell you how many are happening with...something simple. 
"""

# Get today's date
current_date = date.today()

# Print markdown header
md_text = f"""
## {title}
**Description**
{description}
**Data as of**  
{current_date}
"""

# Render it in the output
display(Markdown(md_text))


## 311 Analysis
**Description**

This script will take 311 service requests and tell you how many are happening with...something simple. 

**Data as of**  
2026-09-15


In [ ]:
###########
#PULL DATA#
###########

# Put the URL for your API request here. You can do this using the query builder in ArcGIS Online.
url = "https://services3.arcgis.com/dty2kHktVXHrqO8i/arcgis/rest/services/Data_311/FeatureServer/0/query"

# Type out a where clause here. 
# You can utilize an "f string" to make this filter dynamic.
# NOTE: For many feature layers, the maximum amount of records the ArcGIS Online API can query is 2,000. You'll need to perform multiple queries if you are reading in more than 2k records.
where_clause = """
requested_datetime > '01/01/2026'
"""

# Set query parameters. Nothing here for you to do.
query_params = {
    "where": where_clause,       # The where clause from above.
    "returnGeometry": "false",    # We're not doing any work with spatial data. But if you want to make maps with your data, set to 'true'.
    "f": "json"                  # Tells the server to respond with JSON format.
}

## SUBMIT AND PARSE API REQUEST(S)
# Since in many cases we are limited to reading 2,000 records per query, this function will make several API requests to get all the records.
# It will also parse the request and extract the data into a list of records.
def get_data(url,query_params):
    """
    This function retrieves data from ArcGIS Online FeatureLayer via a series of GET requests. 
    It will pull data in groups of 2,000 records, and append all the data to one list of records.

    Args:
        url (str): Spark session context.
        query_params (dict): A spark DataFrame to geocode.

    Returns:
        list: The spark DataFrame, with new geocoded columns appended.
    """
    # Get record count
    record_count = requests.get(url,params={"where":query_params['where'],"returnCountOnly":"true","f":"json"}).json()['count']

    # Split record count into offsets
    offsets = range(0,record_count,2000)
    
    # List of data records
    results = []
    
    # Loop through offsets and get data for each offset
    for i,offset in enumerate(offsets):
        # Perform a GET request with the given offset
        query_params['resultOffset'] = offset
        req = requests.get(url,params=query_params)
        # Get JSON
        resp = req.json()

        # Extract data and append it to our final result
        data = [a['attributes'] for a in resp['features']]

        results += data
    
    # Ensure the number of records matches the record count of the Feature Layer.
    assert len(results) == record_count

    return results

data = get_data(url, query_params)
data

[{'OBJECTID': 2,
  'service_request_id': '202000405340',
  'address': '1412 E 80th St, Cleveland, OH 44103, US',
  'service_category': 'Illegal Dumping',
  'service_name': 'Illegal Dumping',
  'agency_responsible': 'Public Works',
  'division_responsible': 'Streets',
  'status_description': 'Open',
  'requested_datetime': 1789429621000,
  'updated_datetime': 1789429627000,
  'closed_date': None,
  'target_date': 1789734600000,
  'source': 'Web',
  'parcelpin': '10605028',
  'neighborhood': 'Hough',
  'ward_name': 'Ward 8',
  'ward': 8,
  'ward_name_2014': 'Ward 7',
  'ward_2014': 7,
  'ward_name_2026': 'Ward 8',
  'ward_2026': 8,
  'lat': 41.5167619,
  'long': -81.63327794},
 {'OBJECTID': 5,
  'service_request_id': '202000405337',
  'address': '5103 Luther Ave, Cleveland, OH 44103, US',
  'service_category': 'Illegal Dumping',
  'service_name': 'Illegal Dumping',
  'agency_responsible': 'Public Works',
  'division_responsible': 'Streets',
  'status_description': 'Open',
  'requested_da

In [8]:
######################
#CONVERT TO DATAFRAME#
######################
# If you use the get_data function from above, your data should look something like this:.
"""
[{'service_request_id': '202000403109',
  'service_category': 'Trash & Recycling',
  'service_name': 'Waste Cart Concerns'},
 {'service_request_id': '202000403083',
  'service_category': 'Building & Housing',
  'service_name': 'Electrical Issue'},
 {'service_request_id': '202000403082',
  'service_category': 'Street Issues',
  'service_name': 'Debris in Street'}]
"""

# The format above is known as "records" format, and will allow you to automatically convert to a pandas dataframe when doing pd.DataFrame(records).
# Try converting to DataFrame below:

"\n[{'service_request_id': '202000403109',\n  'service_category': 'Trash & Recycling',\n  'service_name': 'Waste Cart Concerns'},\n {'service_request_id': '202000403083',\n  'service_category': 'Building & Housing',\n  'service_name': 'Electrical Issue'},\n {'service_request_id': '202000403082',\n  'service_category': 'Street Issues',\n  'service_name': 'Debris in Street'}]\n"

In [0]:
##########
#ANALYSIS#
##########

# Conduct your data analysis below. You should use the dataframe from above as your starting point. Feel free to add more cells to separate output.
# We import the "seaborn" package in the first cell above. You can use this or another package of your choosing for creating visualizations.
# If you choose to import additional packages, be sure to update the dependencies in your GitHub Action! Otherwise the workflow will fail.